# Notebook 1: Data Loading & Preprocessing

## Datasets
| # | Name | Task | n | d (raw) |
|---|------|------|---|---|
| 1 | Concrete Compressive Strength | Regression | 1,030 | 8 |
| 2 | Energy Efficiency | Regression | 768 | 8 |
| 3 | Bike Sharing (hourly) | Regression | 17,389 | 12 |
| 4 | Online Shoppers Intent | Classification (binary) | 12,330 | 17 |
| 5 | Statlog Shuttle | Classification (multi-class, 7) | 58,000 | 9 |

All preprocessed data is saved to `data/processed/` for use by other notebooks.


In [5]:

import numpy as np
import pandas as pd
import os
import pickle
import warnings
warnings.filterwarnings('ignore')


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder


from ucimlrepo import fetch_ucirepo


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


os.makedirs('/processed', exist_ok=True)

print("Setup complete. Random seed:", RANDOM_SEED)

Setup complete. Random seed: 42


In [10]:
def preprocess_dataset(X, y, num_cols, cat_cols, task_type, dataset_name, test_size=0.2, val_size=0.2):
   
   
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_SEED,
        stratify=y if task_type == 'classification' else None
    )

    
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_size, random_state=RANDOM_SEED,
        stratify=y_trainval if task_type == 'classification' else None
    )

    
    scaler = StandardScaler()
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    
    if num_cols:
        X_num_train = scaler.fit_transform(X_train[num_cols])
        X_num_val = scaler.transform(X_val[num_cols])
        X_num_test = scaler.transform(X_test[num_cols])
    else:
        X_num_train = np.empty((len(X_train), 0))
        X_num_val = np.empty((len(X_val), 0))
        X_num_test = np.empty((len(X_test), 0))

   
    if cat_cols:
        X_cat_train = ohe.fit_transform(X_train[cat_cols].astype(str))
        X_cat_val = ohe.transform(X_val[cat_cols].astype(str))
        X_cat_test = ohe.transform(X_test[cat_cols].astype(str))
    else:
        X_cat_train = np.empty((len(X_train), 0))
        X_cat_val = np.empty((len(X_val), 0))
        X_cat_test = np.empty((len(X_test), 0))

  
    X_train_final = np.hstack([X_num_train, X_cat_train]).astype(np.float32)
    X_val_final = np.hstack([X_num_val, X_cat_val]).astype(np.float32)
    X_test_final = np.hstack([X_num_test, X_cat_test]).astype(np.float32)

    
    if task_type == 'classification':
        le = LabelEncoder()
        y_train_final = le.fit_transform(y_train).astype(np.int64)
        y_val_final = le.transform(y_val).astype(np.int64)
        y_test_final = le.transform(y_test).astype(np.int64)
        n_classes = len(le.classes_)
    else:
        y_train_final = np.asarray(y_train).astype(np.float32).ravel()
        y_val_final = np.asarray(y_val).astype(np.float32).ravel()
        y_test_final = np.asarray(y_test).astype(np.float32).ravel()
        n_classes = None

    data = {
        'name': dataset_name,
        'task_type': task_type,
        'X_train': X_train_final,
        'X_val': X_val_final,
        'X_test': X_test_final,
        'y_train': y_train_final,
        'y_val': y_val_final,
        'y_test': y_test_final,
        'n_features': X_train_final.shape[1],
        'n_train': len(X_train_final),
        'n_val': len(X_val_final),
        'n_test': len(X_test_final),
        'n_classes': n_classes,
        'num_cols': num_cols,
        'cat_cols': cat_cols,
    }

    print(f"{dataset_name}: n_train={data['n_train']}, n_val={data['n_val']}, n_test={data['n_test']}, d={data['n_features']}, task={task_type}" + (f", classes={n_classes}" if n_classes else ""))
    return data


def save_dataset(data, filename):
    """Save pre/processed data to disk."""
    filepath = f'/processed/{filename}.pkl'
    with open(filepath, 'wb') as f:
        pickle.dump(data, f)
    print(f"saved to {filepath}")


## Dataset 1: Concrete Compressive Strength (Regression)

**Source:** UCI Machine Learning Repository (ID 165)

**Task:** Predict concrete compressive strength (MPa) from 8 ingredient and curing-age features.

**Size:** 1030 samples x 8 features.

**Feature types:** All numerical.

In [8]:
concrete = fetch_ucirepo(id=165)

X_concrete = concrete.data.features.copy()
y_concrete = concrete.data.targets.copy()

y_concrete = y_concrete.iloc[:, 0] if isinstance(y_concrete, pd.DataFrame) else y_concrete

print(f"Shape: {X_concrete.shape}")
print(f"Features: {list(X_concrete.columns)}")
print(f"Target range: [{y_concrete.min():.2f}, {y_concrete.max():.2f}]")
X_concrete.head(3)

Shape: (1030, 8)
Features: ['Cement', 'Blast Furnace Slag', 'Fly Ash', 'Water', 'Superplasticizer', 'Coarse Aggregate', 'Fine Aggregate', 'Age']
Target range: [2.33, 82.60]


,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270


In [11]:
numerical_cols_concrete = list(X_concrete.columns)
categorical_cols_concrete = []

data_concrete = preprocess_dataset(
    X=X_concrete,
    y=y_concrete.astype(np.float32),
    num_cols=numerical_cols_concrete,
    cat_cols=categorical_cols_concrete,
    task_type='regression',
    dataset_name='concrete'
)

save_dataset(data_concrete, 'concrete')

concrete: n_train=659, n_val=165, n_test=206, d=8, task=regression
saved to /processed/concrete.pkl


## Dataset 2: Energy Efficiency (Regression)

**Source:** UCI Machine Learning Repository (ID 242)

**Task:** Predict heating load (Y1) of a building from 8 architectural features.

**Size:** 768 samples × 8 features.

**Feature types:** All numerical.

**Note:** Dataset has two targets (heating load Y1, cooling load Y2). We predict Y1.

In [20]:
energy = fetch_ucirepo(id=242)

X_energy = energy.data.features.copy()
y_energy_full = energy.data.targets.copy()

print(f"Shape: {X_energy.shape}")
print(f"Targets: {list(y_energy_full.columns)} (we use Y1)")
print(f"Features: (denoted by labels X1, X2...)\n{energy.variables['description'].to_string(index=False)}")
y_energy = y_energy_full.iloc[:, 0]

print(f"Target range: [{y_energy.min():.2f}, {y_energy.max():.2f}]")

X_energy.head(3)

Shape: (768, 8)
Targets: ['Y1', 'Y2'] (we use Y1)
Features: (denoted by labels X1, X2...)
     Relative Compactness
             Surface Area
                Wall Area
                Roof Area
           Overall Height
              Orientation
             Glazing Area
Glazing Area Distribution
             Heating Load
             Cooling Load
Target range: [6.01, 43.10]


,X1,X2,X3,X4,X5,X6,X7,X8
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0


In [21]:
numerical_cols_energy = list(X_energy.columns)
categorical_cols_energy = []

data_energy = preprocess_dataset(
    X=X_energy,
    y=y_energy.astype(np.float32),
    num_cols=numerical_cols_energy,
    cat_cols=categorical_cols_energy,
    task_type='regression',
    dataset_name='energy'
)

save_dataset(data_energy, 'energy')

energy: n_train=491, n_val=123, n_test=154, d=8, task=regression
saved to /processed/energy.pkl


## Dataset 3: Bike Sharing (Regression, Large, Mixed Types)

**Source:** UCI Machine Learning Repository (ID 275)

**Task:** Predict the hourly count of rented bikes from temporal and weather features.

**Size:** 17,389 samples × 11 features (after dropping date which is used for event detection task, not count prediction task).

**Feature types:** **Mixed** numerical (temp, humidity, windspeed) + categorical (season, weekday, hour, weather situation).


In [24]:
bike = fetch_ucirepo(id=275)

X_bike = bike.data.features.copy()
y_bike = bike.data.targets.copy()

y_bike = y_bike.iloc[:, 0] if isinstance(y_bike, pd.DataFrame) else y_bike

drop_cols = [c for c in ['instant', 'dteday', 'casual', 'registered', 'yr'] if c in X_bike.columns]
if drop_cols:
    X_bike = X_bike.drop(columns=drop_cols)
    print(f"\nDropped features not useful for our task: {drop_cols}")

print(f"Shape: {X_bike.shape}")
print(f"Features: {list(X_bike.columns)}")
print(f"Target range: [{y_bike.min()}, {y_bike.max()}]")
print(f"\nDtypes:")
print(X_bike.dtypes)
X_bike.head(3)


Dropped features not useful for our task: ['dteday', 'yr']
Shape: (17379, 11)
Features: ['season', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
Target range: [1, 977]

Dtypes:
season          int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
dtype: object


,season,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,1,1,0,0,6,0,1,0.24,0.2879,0.81,0.0
1,1,1,1,0,6,0,1,0.22,0.2727,0.80,0.0
2,1,1,2,0,6,0,1,0.22,0.2727,0.80,0.0


In [26]:
cat_cols_bike = ['season', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
num_cols_bike = [c for c in X_bike.columns if c not in cat_cols_bike]

print(f"Numerical columns ({len(num_cols_bike)}): {num_cols_bike}")
print(f"Categorical columns ({len(cat_cols_bike)}): {cat_cols_bike}")

for c in cat_cols_bike:
    X_bike[c] = X_bike[c].astype(str)

data_bike = preprocess_dataset(
    X=X_bike,
    y=y_bike.astype(np.float32),
    num_cols=num_cols_bike,
    cat_cols=cat_cols_bike,
    task_type='regression',
    dataset_name='bike_sharing'
)

save_dataset(data_bike, 'bike_sharing')

Numerical columns (4): ['temp', 'atemp', 'hum', 'windspeed']
Categorical columns (7): ['season', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit']
bike_sharing: n_train=11122, n_val=2781, n_test=3476, d=59, task=regression
saved to /processed/bike_sharing.pkl


## Dataset 4: Online Shoppers Purchasing Intention (Classification, Binary, Mixed Types)

**Source:** UCI Machine Learning Repository (ID 468)

**Task:** Predict whether a website session ends in a purchase (Revenue = True/False) from behavioural and contextual session features.

**Size:** ~12,330 samples × 17 features.

**Feature types:** **Mixed** numerical (page durations, bounce rates, exit rates, page values) + categorical (Month, OperatingSystems, Browser, Region, TrafficType, VisitorType, Weekend).

In [28]:
shoppers = fetch_ucirepo(id=468)

X_shop = shoppers.data.features.copy()
y_shop = shoppers.data.targets.copy()

y_shop = y_shop.iloc[:, 0] if isinstance(y_shop, pd.DataFrame) else y_shop
y_shop = y_shop.astype(str).str.strip()

print(f"Shape: {X_shop.shape}")
print(f"Features:")
print(X_shop.dtypes)

X_shop.head(3)

Shape: (12330, 17)
Features:
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
dtype: object


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False
2,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False


In [30]:
cat_cols_shop = ['Weekend', 'VisitorType', 'TrafficType', 'Region', 'Browser', 'OperatingSystems', 'Month']
num_cols_shop = [c for c in X_shop.columns if c not in cat_cols_shop]

print(f"Numerical columns ({len(num_cols_shop)}): {num_cols_shop}")
print(f"Categorical columns ({len(cat_cols_shop)}): {cat_cols_shop}")

for c in cat_cols_shop:
    X_shop[c] = X_shop[c].fillna('unknown').astype(str)

data_shop = preprocess_dataset(
    X=X_shop,
    y=y_shop,
    num_cols=num_cols_shop,
    cat_cols=cat_cols_shop,
    task_type='classification',
    dataset_name='online_shoppers'
)

save_dataset(data_shop, 'online_shoppers')

Numerical columns (10): ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
Categorical columns (7): ['Weekend', 'VisitorType', 'TrafficType', 'Region', 'Browser', 'OperatingSystems', 'Month']
online_shoppers: n_train=7891, n_val=1973, n_test=2466, d=75, task=classification, classes=2
saved to /processed/online_shoppers.pkl


## Dataset 5: Statlog Shuttle (Classification, Multi-Class, Large)

**Source:** UCI Machine Learning Repository (ID 148)

**Task:** Classify NASA shuttle telemetry into one of 7 operational states from 9 numerical sensor readings.

**Size:** 58,000 samples × 9 features.

**Feature Types:** All numerical.

In [32]:
shuttle = fetch_ucirepo(id=148)

X_shuttle = shuttle.data.features.copy()
y_shuttle = shuttle.data.targets.copy()

y_shuttle = y_shuttle.iloc[:, 0] if isinstance(y_shuttle, pd.DataFrame) else y_shuttle

print(f"Shape: {X_shuttle.shape}")
print(f"Features: {list(X_shuttle.columns)}")
print(f"Total classes: {y_shuttle.nunique()}")
X_shuttle.head(3)

Shape: (58000, 7)
Features: ['Rad Flow', 'Fpv Close', 'Fpv Open', 'High', 'Bypass', 'Bpv Close', 'Bpv Open']
Total classes: 7


,,Rad Flow,Fpv Close,Fpv Open,High,Bypass,Bpv Close,Bpv Open
50,21,77,0,28,0,27,48,22
55,0,92,0,0,26,36,92,56
53,0,82,0,52,-5,29,30,2


In [33]:
num_cols_shuttle = list(X_shuttle.columns)
cat_cols_shuttle = []

data_shuttle = preprocess_dataset(
    X=X_shuttle,
    y=y_shuttle,
    num_cols=num_cols_shuttle,
    cat_cols=cat_cols_shuttle,
    task_type='classification',
    dataset_name='shuttle'
)

save_dataset(data_shuttle, 'shuttle')

shuttle: n_train=37120, n_val=9280, n_test=11600, d=7, task=classification, classes=7
saved to /processed/shuttle.pkl
